# Freight Rate Prediction — Spotter ML Engineer Assessment

This notebook is the end-to-end, narrative version of the pipeline in `src/`
(`data_prep.py`, `train.py`, `predict.py`, `eda.py`) — same logic, same
model, same outputs, but with the exploration, reasoning, and results shown
inline so it's easy to read top to bottom.

**Contents**
1. Load & inspect the data
2. Data-quality issues found
3. Feature engineering
4. Train / validation split strategy
5. Baseline model training & evaluation
6. Hyperparameter fine-tuning
7. Refit the tuned model on all labeled data
8. Score `validation.csv` → `validation_predictions.csv`
9. Score the December fixed-lane chart inputs
10. Run the provided `score.py`

Running `python -m src.train` and `python -m src.predict` from the command
line does exactly what this notebook does, for the actual submission
files — this notebook is for exploration/walkthrough.


## 0. Setup

In [ ]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_absolute_percentage_error, r2_score

pd.set_option("display.max_columns", None)
RANDOM_STATE = 42

DATA_DIR = Path("data")
OUTPUT_DIR = Path("outputs")
MODEL_DIR = Path("models")
OUTPUT_DIR.mkdir(exist_ok=True)
MODEL_DIR.mkdir(exist_ok=True)


## 1. Load & inspect the data

`train_test.csv` is the only labeled file (48,000 rows). It has load
identifiers, origin/destination cities with lat/lon, distance, equipment
type, weight, date, two market signals (`market_index`, `quote_signal`),
and the target, `posted_rate`.

In [ ]:
train = pd.read_csv(DATA_DIR / "train_test.csv")
validation = pd.read_csv(DATA_DIR / "validation.csv")
template = pd.read_csv(DATA_DIR / "validation_predictions_template.csv")
december = pd.read_csv(DATA_DIR / "december_chart_inputs.csv")

print("train_test.csv:", train.shape)
print("validation.csv:", validation.shape)
print("validation_predictions_template.csv:", template.shape)
print("december_chart_inputs.csv:", december.shape)
train.head()


In [ ]:
train.dtypes

In [ ]:
train.describe()

In [ ]:
print("Date range (train):", train["date"].min(), "to", train["date"].max())
print("Date range (validation):", validation["date"].min(), "to", validation["date"].max())
print("Date range (december):", december["date"].min(), "to", december["date"].max())


**Key finding #1:** `train_test.csv` covers **2025-01-01 to 2025-10-31**.
`validation.csv` and the December file are **entirely in the future**
relative to that window (Nov–Dec 2025). This one fact drives the split
strategy in Section 4 — a random split would understate how hard the real
task is.

## 2. Data-quality issues found

In [ ]:
train.isna().sum()[train.isna().sum() > 0]

In [ ]:
n_missing_weight = train["weight"].isna().sum()
n_negative_weight = (train["weight"] < 0).sum()
n_missing_market = train["market_index"].isna().sum()

print(f"weight: {n_missing_weight} missing, {n_negative_weight} negative "
      f"({(n_missing_weight + n_negative_weight) / len(train) * 100:.2f}% of rows affected)")
print(f"market_index: {n_missing_market} missing ({n_missing_market / len(train) * 100:.2f}%)")

train.loc[train["weight"] < 0, ["load_id", "weight"]].head()


**Issue A — negative `weight`:** ~0.6% of rows have negative weight, which
is physically impossible for a load. Treated as a sign-entry error and
corrected with `abs()` rather than dropped (dropping would throw away
otherwise-valid distance/rate information, and every row needs a
prediction anyway for `validation.csv`).

**Issue B — missing `weight`:** a further ~0.6% of rows are missing weight
entirely. Imputed with the **training-set** median weight for that row's
equipment type (weight distributions differ meaningfully by equipment —
see below), never using validation/December statistics.

**Issue C — missing `market_index`:** ~0.8% of rows. Left as `NaN` —
`HistGradientBoostingRegressor` handles missing numeric values natively,
and `market_index` correlates only weakly with the target anyway (checked
next), so imputing it would add little.

In [ ]:
train.groupby("equipment")["weight"].median()

In [ ]:
numeric_cols = ["distance", "weight", "market_index", "quote_signal", "posted_rate"]
train[numeric_cols].corr()["posted_rate"].sort_values(ascending=False)


`distance` is overwhelmingly the strongest single predictor
(r ≈ 0.91). `weight`, `market_index`, and `quote_signal` each correlate
weakly (|r| < 0.1) individually — but a gradient-boosted model can still
combine them with equipment/lane/date for real lift (see Section 5).

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
ax.scatter(train["distance"], train["posted_rate"], s=4, alpha=0.25, color="#064A56")
ax.set_xlabel("distance (miles)")
ax.set_ylabel("posted_rate ($)")
ax.set_title("posted_rate vs distance")
plt.tight_layout()
plt.show()


In [ ]:
train["rate_per_mile"] = train["posted_rate"] / train["distance"]
train.groupby("equipment")["rate_per_mile"].describe()[["mean", "50%", "std"]]


**Key finding #2:** Reefer and Flatbed loads command a meaningfully
higher `$/mile` than Dry Van on average — equipment type matters and is
included as a categorical feature.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
train.boxplot(column="rate_per_mile", by="equipment", ax=ax)
ax.set_ylim(0, train["rate_per_mile"].quantile(0.99))
ax.set_title("$/mile by equipment type")
plt.suptitle("")
plt.tight_layout()
plt.show()


In [ ]:
train_cities = set(train["pickup"]) | set(train["delivery"])
val_cities = set(validation["pickup"]) | set(validation["delivery"])
unseen_cities = val_cities - train_cities

train_lanes = set(zip(train["pickup"], train["delivery"]))
val_lanes = set(zip(validation["pickup"], validation["delivery"]))
unseen_lanes = val_lanes - train_lanes

print(f"Cities in validation.csv never seen in training: {sorted(unseen_cities)}")
print(f"Lanes in validation.csv never seen in training: {len(unseen_lanes)} of {len(val_lanes)}")


**Issue D — unseen cities/lanes in `validation.csv`:** 8 pickup/delivery
cities and 736 of 4,214 lanes never appear in training. This drove two
modeling choices (Section 3): `pickup`/`delivery` are encoded as native
categoricals so unseen levels degrade gracefully instead of crashing, and
lat/lon are included as numeric features so the model still has a
geography signal for brand-new cities.

In [ ]:
monthly = train.copy()
monthly["date"] = pd.to_datetime(monthly["date"])
monthly_rpm = monthly.groupby(monthly["date"].dt.to_period("M"))["rate_per_mile"].mean()

fig, ax = plt.subplots(figsize=(7, 5))
monthly_rpm.plot(kind="bar", ax=ax, color="#064A56")
ax.set_title("Mean $/mile by month (2025 training data)")
ax.set_ylabel("$/mile")
plt.tight_layout()
plt.show()


**Key finding #3:** no strong single-direction trend in monthly `$/mile`
across the training window. This matters for the December chart
(Section 9): there's no evidence-based trend to extrapolate `market_index`
/ `quote_signal` forward into December, so a neutral placeholder is used
instead of a guessed trend.

## 3. Feature engineering

- **Numeric:** `distance`, `weight` (cleaned), `market_index`, `quote_signal`,
  `pickup_lat`, `pickup_lon`, `delivery_lat`, `delivery_lon`
- **Date-derived:** `month`, `day_of_week`, `is_weekend`, and
  `day_of_year` encoded as `sin`/`cos` (cyclical — so Dec 31 and Jan 1 are
  correctly "close" instead of 364 days apart)
- **Categorical (native, not one-hot):** `equipment`, `pickup`, `delivery`

All of this is implemented once in `src/data_prep.py` and reused for
training and both prediction files, so the notebook and the `.py`
pipeline can never drift out of sync.

In [ ]:
TARGET_COL = "posted_rate"
CATEGORICAL_COLS = ["equipment", "pickup", "delivery"]
BASE_NUMERIC_COLS = ["distance", "weight", "market_index", "quote_signal",
                     "pickup_lat", "pickup_lon", "delivery_lat", "delivery_lon"]
DATE_FEATURE_COLS = ["month", "day_of_week", "is_weekend", "day_of_year_sin", "day_of_year_cos"]
FEATURE_COLS = BASE_NUMERIC_COLS + DATE_FEATURE_COLS + CATEGORICAL_COLS


def clean_weight(df, fallback_medians=None):
    df = df.copy()
    df["weight"] = df["weight"].astype(float).abs()
    if fallback_medians is not None:
        global_median = fallback_medians["__global__"]
        for equipment, median_value in fallback_medians.items():
            if equipment == "__global__":
                continue
            mask = (df["equipment"] == equipment) & (df["weight"].isna())
            df.loc[mask, "weight"] = median_value
        df["weight"] = df["weight"].fillna(global_median)
    return df


def compute_weight_medians(train_df):
    cleaned = train_df.copy()
    cleaned["weight"] = cleaned["weight"].astype(float).abs()
    medians = cleaned.groupby("equipment")["weight"].median().to_dict()
    medians["__global__"] = cleaned["weight"].median()
    return medians


def add_date_features(df):
    df = df.copy()
    df["date"] = pd.to_datetime(df["date"])
    day_of_year = df["date"].dt.dayofyear
    df["month"] = df["date"].dt.month
    df["day_of_week"] = df["date"].dt.dayofweek
    df["is_weekend"] = (df["day_of_week"] >= 5).astype(int)
    df["day_of_year_sin"] = np.sin(2 * np.pi * day_of_year / 365.25)
    df["day_of_year_cos"] = np.cos(2 * np.pi * day_of_year / 365.25)
    return df


def get_category_levels(train_df):
    return {col: sorted(train_df[col].dropna().unique().tolist()) for col in CATEGORICAL_COLS}


def set_categorical_dtypes(df, category_levels):
    df = df.copy()
    for col in CATEGORICAL_COLS:
        df[col] = pd.Categorical(df[col], categories=category_levels[col])
    return df


def prepare_features(df, weight_medians, category_levels):
    df = clean_weight(df, fallback_medians=weight_medians)
    df = add_date_features(df)
    df = set_categorical_dtypes(df, category_levels)
    return df[FEATURE_COLS]


## 4. Train / validation split strategy

**Time-based, not random.** `train_test.csv` covers Jan–Oct 2025, while
`validation.csv` and the December file are entirely *future* dates
(Nov–Dec 2025). A random split would let the model train on rows
interleaved throughout the year and be evaluated on a random sample from
the same period — that overstates real-world generalization, because the
actual task is "predict rates for dates after the training window ends."

The holdout here is the **last ~15% of days** in `train_test.csv`
(2025-09-15 onward), so the holdout mimics the real task as closely as
possible using only the labeled data we have.

In [ ]:
def time_based_split(df, holdout_fraction=0.15):
    df = df.copy()
    df["date"] = pd.to_datetime(df["date"])
    df = df.sort_values("date")
    cutoff_index = int(len(df) * (1 - holdout_fraction))
    cutoff_date = df.iloc[cutoff_index]["date"]
    train_part = df[df["date"] < cutoff_date].reset_index(drop=True)
    holdout_part = df[df["date"] >= cutoff_date].reset_index(drop=True)
    return train_part, holdout_part, cutoff_date


train_part, holdout_part, cutoff_date = time_based_split(train, holdout_fraction=0.15)
print(f"train: {len(train_part):,} rows (< {cutoff_date.date()})")
print(f"holdout: {len(holdout_part):,} rows (>= {cutoff_date.date()})")


## 5. Baseline model training & evaluation

**Model:** `HistGradientBoostingRegressor` (scikit-learn) — handles the
mix of numeric and native categorical features without one-hot encoding
or scaling, and copes gracefully with missing values and categorical
levels unseen at predict time (both of which are present here).

**Target transform:** `log1p(posted_rate)`. The target is right-skewed
(median ≈ $2,031, max ≈ $25,533) and its spread scales with distance —
modeling the log keeps a handful of very expensive long-haul loads from
dominating the loss, and predictions are inverted with `expm1`.

In [ ]:
weight_medians = compute_weight_medians(train_part)
category_levels = get_category_levels(train_part)

X_train = prepare_features(train_part, weight_medians, category_levels)
X_holdout = prepare_features(holdout_part, weight_medians, category_levels)
y_train_log = np.log1p(train_part[TARGET_COL].values)
y_holdout_log = np.log1p(holdout_part[TARGET_COL].values)

categorical_feature_idx = [i for i, c in enumerate(FEATURE_COLS) if c in CATEGORICAL_COLS]

model = HistGradientBoostingRegressor(
    loss="squared_error",
    max_iter=600,
    learning_rate=0.05,
    max_leaf_nodes=31,
    min_samples_leaf=25,
    l2_regularization=0.1,
    early_stopping=True,
    n_iter_no_change=25,
    validation_fraction=0.1,
    categorical_features=categorical_feature_idx,
    random_state=RANDOM_STATE,
)
model.fit(X_train, y_train_log)
print("Trained. n_iter_:", model.n_iter_)


In [ ]:
holdout_pred_log = model.predict(X_holdout)
holdout_pred = np.clip(np.expm1(holdout_pred_log), 1.0, None)
holdout_true = np.expm1(y_holdout_log)

metrics = {
    "mae_usd": mean_absolute_error(holdout_true, holdout_pred),
    "mape_pct": mean_absolute_percentage_error(holdout_true, holdout_pred) * 100,
    "r2": r2_score(holdout_true, holdout_pred),
    "n_holdout_rows": len(holdout_true),
}
for k, v in metrics.items():
    print(f"{k}: {v:,.4f}" if isinstance(v, float) else f"{k}: {v}")


In [ ]:
# Sanity-check baseline: naive "distance x training-mean $/mile" heuristic
train_rpm = (train_part[TARGET_COL] / train_part["distance"]).mean()
baseline_pred = holdout_part["distance"].values * train_rpm
baseline_mae = mean_absolute_error(holdout_part[TARGET_COL].values, baseline_pred)
baseline_mape = mean_absolute_percentage_error(holdout_part[TARGET_COL].values, baseline_pred) * 100

print(f"Baseline (distance x avg $/mile)  MAE: ${baseline_mae:,.2f}   MAPE: {baseline_mape:.2f}%")
print(f"Model                              MAE: ${metrics['mae_usd']:,.2f}   MAPE: {metrics['mape_pct']:.2f}%")
print(f"Improvement over baseline: {(1 - metrics['mae_usd'] / baseline_mae) * 100:.1f}% lower MAE")


In [ ]:
fig, ax = plt.subplots(figsize=(6, 6))
ax.scatter(holdout_true, holdout_pred, s=6, alpha=0.3, color="#064A56")
lims = [0, max(holdout_true.max(), holdout_pred.max())]
ax.plot(lims, lims, color="red", linewidth=1, linestyle="--")
ax.set_xlabel("actual posted_rate ($)")
ax.set_ylabel("predicted posted_rate ($)")
ax.set_title("Holdout: predicted vs. actual")
plt.tight_layout()
plt.show()


## 6. Hyperparameter fine-tuning

The Section 5 model used reasonable default settings. Now we search over
`HistGradientBoostingRegressor`'s most impactful knobs — `learning_rate`,
`max_leaf_nodes`, `min_samples_leaf`, `l2_regularization` — to see if a
better combination exists.

**Important:** every candidate is scored on the *same time-based holdout*
from Section 4, not a random K-fold. Tuning against a random split would
reward configurations that overfit to patterns that happen to repeat
within the training year, which is exactly the mistake Section 4 was
designed to avoid — the holdout has to keep mimicking "predict the
future" or the tuning step would quietly undo the point of the split.

In [ ]:
param_space = {
    "learning_rate": [0.03, 0.05, 0.08, 0.1],
    "max_leaf_nodes": [15, 31, 63, 127],
    "min_samples_leaf": [10, 20, 25, 40, 60],
    "l2_regularization": [0.0, 0.1, 0.3, 0.5, 1.0],
}

N_TRIALS = 16
rng = np.random.RandomState(RANDOM_STATE)
y_holdout = np.expm1(y_holdout_log)

trial_results = []
for trial in range(N_TRIALS):
    params = {k: rng.choice(v) for k, v in param_space.items()}
    params = {k: (float(v) if isinstance(v, np.floating) else int(v)) for k, v in params.items()}

    candidate = HistGradientBoostingRegressor(
        loss="squared_error", max_iter=600, early_stopping=True,
        n_iter_no_change=25, validation_fraction=0.1,
        categorical_features=categorical_feature_idx, random_state=RANDOM_STATE,
        **params,
    )
    candidate.fit(X_train, y_train_log)
    pred = np.clip(np.expm1(candidate.predict(X_holdout)), 1.0, None)

    trial_results.append({
        **params,
        "mae": mean_absolute_error(y_holdout, pred),
        "mape": mean_absolute_percentage_error(y_holdout, pred) * 100,
        "r2": r2_score(y_holdout, pred),
        "n_iter": candidate.n_iter_,
    })

results_df = pd.DataFrame(trial_results).sort_values("mae").reset_index(drop=True)
results_df


In [ ]:
best_params = results_df.iloc[0][["learning_rate", "max_leaf_nodes", "min_samples_leaf", "l2_regularization"]].to_dict()
best_params = {
    "learning_rate": float(best_params["learning_rate"]),
    "max_leaf_nodes": int(best_params["max_leaf_nodes"]),
    "min_samples_leaf": int(best_params["min_samples_leaf"]),
    "l2_regularization": float(best_params["l2_regularization"]),
}

print("Best config found:", best_params)
print(f"Holdout MAE:  baseline ${metrics['mae_usd']:.2f}  ->  tuned ${results_df.iloc[0]['mae']:.2f}")
print(f"Holdout MAPE: baseline {metrics['mape_pct']:.2f}%  ->  tuned {results_df.iloc[0]['mape']:.2f}%")
print(f"Holdout R2:   baseline {metrics['r2']:.4f}  ->  tuned {results_df.iloc[0]['r2']:.4f}")


The search finds a modest improvement, not a dramatic one — expected,
since `distance` already explains most of the variance (Section 2) and
the default settings in Section 5 weren't far off. The tuned
hyperparameters are what get used for the final model below. If this
were going into production, the natural next step is a wider/longer
search (e.g. `RandomizedSearchCV`-style with more trials, or Optuna) —
noted in the report's limitations section.

## 7. Refit the tuned model on all labeled data

After confirming the tuned config on the time-based holdout above, the
final model is refit on *all* 48,000 rows of `train_test.csv` (train +
holdout combined) using the winning hyperparameters, before scoring the
real `validation.csv` and December files — so no labeled data goes
unused in the shipped model.

In [ ]:
full_weight_medians = compute_weight_medians(train)
full_category_levels = get_category_levels(train)

X_full = prepare_features(train, full_weight_medians, full_category_levels)
y_full_log = np.log1p(train[TARGET_COL].values)

final_model = HistGradientBoostingRegressor(
    loss="squared_error",
    max_iter=600,
    early_stopping=True,
    n_iter_no_change=25,
    validation_fraction=0.1,
    categorical_features=categorical_feature_idx,
    random_state=RANDOM_STATE,
    **best_params,
)
final_model.fit(X_full, y_full_log)
print("Final tuned model trained on all", len(train), "rows. n_iter_:", final_model.n_iter_)

# Persist tuned holdout metrics (Section 6's winning trial) alongside the baseline ones
tuned_metrics = results_df.iloc[0].to_dict()
tuned_metrics["best_params"] = best_params
with open(OUTPUT_DIR / "holdout_metrics.json", "w") as f:
    json.dump({"baseline": metrics, "tuned": tuned_metrics}, f, indent=2, default=float)


## 8. Score `validation.csv` → `validation_predictions.csv`

12,000 loads, each with a unique `load_id`. Predictions are written into
the structure of `validation_predictions_template.csv` so the column
order/row order exactly matches what `score.py` expects.

In [ ]:
assert (template["load_id"].values == validation["load_id"].values).all(), \
    "template and validation.csv load_id order must match"

X_val = prepare_features(validation, full_weight_medians, full_category_levels)
val_pred = np.clip(np.expm1(final_model.predict(X_val)), 1.0, None)

validation_predictions = template.copy()
validation_predictions["predicted_rate"] = np.round(val_pred, 2)
validation_predictions.to_csv(OUTPUT_DIR / "validation_predictions.csv", index=False)

print(validation_predictions.shape)
validation_predictions.head()


In [ ]:
validation_predictions["predicted_rate"].describe()

## 9. Score the December fixed-lane chart inputs

`december_chart_inputs.csv` fixes `pickup` (Lexington), `delivery` (Fort
Wayne), `distance` (360 mi), `equipment` (Dry Van), and `weight`
(32,000 lb) — only `date` varies across the 31 days of December 2025.

This file is missing lat/lon and `market_index`/`quote_signal` columns
entirely (unlike `train_test.csv`/`validation.csv`), so they're
reconstructed first:
- **lat/lon:** exact lookup from a city→coordinate table built from
  training data (every city has one fixed coordinate pair, verified in
  Section 2).
- **`market_index`/`quote_signal`:** filled with their **training-set
  global mean**. Both correlate weakly with `posted_rate` (Section 2) and
  showed no consistent month-over-month trend to extrapolate (Section 2,
  Key finding #3) — a neutral, evidence-based placeholder beats an
  unsupported trend guess. The resulting curve is driven by what actually
  varies here: date-based seasonality.

In [ ]:
def build_city_coords(train_df):
    pickup_coords = train_df[["pickup", "pickup_lat", "pickup_lon"]].rename(
        columns={"pickup": "city", "pickup_lat": "lat", "pickup_lon": "lon"})
    delivery_coords = train_df[["delivery", "delivery_lat", "delivery_lon"]].rename(
        columns={"delivery": "city", "delivery_lat": "lat", "delivery_lon": "lon"})
    all_coords = pd.concat([pickup_coords, delivery_coords], ignore_index=True).drop_duplicates("city")
    return {row.city: (row.lat, row.lon) for row in all_coords.itertuples()}


def fill_missing_market_columns(df, city_coords, market_index_default, quote_signal_default):
    df = df.copy()
    if "pickup_lat" not in df.columns:
        df["pickup_lat"] = df["pickup"].map(lambda c: city_coords.get(c, (np.nan, np.nan))[0])
        df["pickup_lon"] = df["pickup"].map(lambda c: city_coords.get(c, (np.nan, np.nan))[1])
    if "delivery_lat" not in df.columns:
        df["delivery_lat"] = df["delivery"].map(lambda c: city_coords.get(c, (np.nan, np.nan))[0])
        df["delivery_lon"] = df["delivery"].map(lambda c: city_coords.get(c, (np.nan, np.nan))[1])
    if "market_index" not in df.columns:
        df["market_index"] = market_index_default
    if "quote_signal" not in df.columns:
        df["quote_signal"] = quote_signal_default
    return df


city_coords = build_city_coords(train)
market_index_default = float(train["market_index"].mean())
quote_signal_default = float(train["quote_signal"].mean())

december_enriched = fill_missing_market_columns(december, city_coords, market_index_default, quote_signal_default)
X_dec = prepare_features(december_enriched, full_weight_medians, full_category_levels)
dec_pred = np.clip(np.expm1(final_model.predict(X_dec)), 1.0, None)

# Output keeps the ORIGINAL 7 columns only (score.py requires exact column order)
december_filled = december.copy()
december_filled["predicted_rate"] = np.round(dec_pred, 2)
december_filled.to_csv(OUTPUT_DIR / "december_chart_inputs_filled.csv", index=False)
december_filled


In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(pd.to_datetime(december_filled["date"]), december_filled["predicted_rate"],
        marker="o", markersize=3, color="#064A56")
ax.set_title("December 2025 predicted rate — Lexington to Fort Wayne (preview)")
ax.set_ylabel("predicted_rate ($)")
plt.xticks(rotation=35)
plt.tight_layout()
plt.show()


**Key finding #4:** even with market signals held neutral, the model
recovers a **weekly (day-of-week) cycle** learned from training-data
patterns — visible as the small repeating dips/peaks above — despite
December being entirely outside the training date range.

## 10. Run the provided `score.py`

`score.py` validates both output files and produces the official
`scorer_results/candidate_december.png` chart used in the report. Run it
from a terminal in the project root (it's a `SystemExit`-based CLI script,
not meant to be imported):

```bash
python score.py --predictions outputs/validation_predictions.csv \
                 --december-predictions outputs/december_chart_inputs_filled.csv
```

Expected output:

```
Validated 12,000 final predictions.
Validated 31 fixed December predictions.
Created chart: scorer_results/candidate_december.png
```

The chart itself, plus the full write-up of every decision made in this
notebook, is in `report/Freight_Rate_Assessment_Report.docx`.